# 03b — QUBO Design for Scenario-Based Trial Selection

This notebook translates the Scenario B trial selection problem into a
Quadratic Unconstrained Binary Optimization (QUBO) formulation.

We build on Phase 2 and 03a:

- Scenario tables:
  - `data/scenarios/scenario_A_trials.csv`
  - `data/scenarios/scenario_B_trials.csv`
- Greedy baseline solution:
  - `data/scenarios/scenario_B_greedy_selection.csv`

Decision problem (for a chosen scenario, default: B):

> Select a subset of trials to **maximize total benefit_score**  
> subject to **total estimated_trial_cost ≤ budget**.

QUBO idea (minimization):

- Binary variable \( x_i \in \{0, 1\} \) indicates whether trial *i* is selected.
- We want to:
  - Maximize \( \sum_i b_i x_i \) (benefit), and
  - Enforce budget \( \sum_i c_i x_i \le B \) (cost ≤ budget).
- QUBO objective:
  \[
  \min_x \left(
      - \sum_i b_i x_i
      + \lambda \, (\sum_i c_i x_i - B)^2
  \right)
  \]
where \( \lambda > 0 \) is a penalty weight.

This notebook:

1. Loads Scenario B and optionally restricts to a manageable subset of trials.
2. Builds the QUBO coefficients from benefit, cost, and budget.
3. Saves the resulting QUBO in a simple dictionary format for later use in
   quantum / hybrid experiments.


In [1]:
# ============================================================
# Cell 1 — Load Scenario B + greedy solution and select subset
# ============================================================

from pathlib import Path
import pandas as pd

def log(msg: str) -> None:
    print(msg)

SCENARIO_B_PATH = Path("data/scenarios/scenario_B_trials.csv")
GREEDY_SEL_PATH = Path("data/scenarios/scenario_B_greedy_selection.csv")

if not SCENARIO_B_PATH.exists():
    raise FileNotFoundError(
        f"[Cell 1] Missing {SCENARIO_B_PATH}. "
        "Run 02d_define_scenarios_A_B.ipynb first."
    )

scenario_B = pd.read_csv(SCENARIO_B_PATH)
log(f"[Cell 1] Loaded Scenario B with shape {scenario_B.shape}")

greedy_selection = None
if GREEDY_SEL_PATH.exists():
    greedy_selection = pd.read_csv(GREEDY_SEL_PATH)
    log(
        f"[Cell 1] Loaded greedy baseline selection with shape "
        f"{greedy_selection.shape}"
    )
else:
    log("[Cell 1] WARNING: No greedy baseline selection found; continuing without it.")

# For QUBO experiments, we typically restrict to a smaller subset of trials
# so that the number of binary variables is manageable.
#
# Here we pick the top N trials by benefit_score within Scenario B.
MAX_TRIALS = 40  # you can adjust this as needed

scenario_B_sorted = scenario_B.sort_values(
    "benefit_score", ascending=False
).reset_index(drop=True)

qubo_candidates = scenario_B_sorted.head(MAX_TRIALS).reset_index(drop=True)
log(
    f"[Cell 1] Using top {len(qubo_candidates)} trials by benefit_score "
    f"for QUBO design."
)

qubo_candidates[["nct_id", "phase", "overall_status",
                 "estimated_trial_cost", "benefit_score"]].head(10)


[Cell 1] Loaded Scenario B with shape (10268, 13)
[Cell 1] Loaded greedy baseline selection with shape (1026, 14)
[Cell 1] Using top 40 trials by benefit_score for QUBO design.


,nct_id,phase,overall_status,estimated_trial_cost,benefit_score
0,NCT00003042,Phase 2,"Active, not recruiting",2.0,0.6
1,NCT06234631,Phase 2,Recruiting,2.0,0.6
2,NCT06257537,Phase 2,Recruiting,2.0,0.6
3,NCT06257875,Phase 2,"Active, not recruiting",2.0,0.6
4,NCT06259123,Phase 2,Recruiting,2.0,0.6
5,NCT06259721,Phase 2,Recruiting,2.0,0.6
6,NCT06259929,Phase 2,Recruiting,2.0,0.6
7,NCT06260033,Phase 2,Recruiting,2.0,0.6
8,NCT06260072,Phase 2,Recruiting,2.0,0.6
9,NCT06260553,Phase 2,Enrolling by invitation,2.0,0.6


### What Cell 1 Just Did

This step prepared the candidate trials that will be encoded into the QUBO.

It:

- Loaded `Scenario B` from `data/scenarios/scenario_B_trials.csv`.
- Optionally loaded the greedy baseline selection from
  `data/scenarios/scenario_B_greedy_selection.csv` (for later comparison).
- Sorted Scenario B trials by `benefit_score` in descending order.
- Selected the top `MAX_TRIALS` (default 40) as `qubo_candidates`.

These `qubo_candidates` define the **binary decision variables** for the QUBO:
each candidate trial *i* will correspond to a binary variable \( x_i \in \{0, 1\} \).


In [3]:
# ============================================================
# Cell 2 — Build QUBO dictionary for trial selection
# ============================================================
#
# QUBO form:
#   Minimize  - sum_i b_i x_i  +  λ (sum_i c_i x_i - B)^2
#
# Expanding the squared term:
#   (sum_i c_i x_i - B)^2
#   = sum_i c_i^2 x_i^2 + 2 sum_{i<j} c_i c_j x_i x_j - 2 B sum_i c_i x_i + B^2
#
# Since x_i^2 = x_i for binary variables:
#   Objective (up to constant B^2) becomes:
#
#   ∑_i [ -b_i + λ (c_i^2 - 2 B c_i) ] x_i  +  2 λ ∑_{i<j} c_i c_j x_i x_j
#
# We encode this as a QUBO dictionary Q[(i, j)].

import numpy as np

required_cols = ["estimated_trial_cost", "benefit_score"]
missing = [c for c in required_cols if c not in qubo_candidates.columns]
if missing:
    raise ValueError(f"[Cell 2] Missing required columns in qubo_candidates: {missing}")

# Extract cost and benefit as numpy arrays
costs = qubo_candidates["estimated_trial_cost"].to_numpy(dtype=float)
benefits = qubo_candidates["benefit_score"].fillna(0.0).to_numpy(dtype=float)

N = len(qubo_candidates)
log(f"[Cell 2] Building QUBO for N = {N} binary variables.")

# Budget: we can reuse the idea from 03a, e.g., a fraction of total cost
total_cost = costs.sum()
BUDGET = 0.1 * total_cost  # 10% of total cost; adjust as needed

# Penalty weight λ: should be large enough that violating the budget
# is "more expensive" than any realistic benefit gain.
lambda_penalty = 10.0

log(f"[Cell 2] Using budget BUDGET = {BUDGET:,.2f}")
log(f"[Cell 2] Using penalty λ = {lambda_penalty:,.2f}")

Q = {}  # QUBO dictionary: keys are (i, j), values are coefficients

# Diagonal terms: i == j
for i in range(N):
    c_i = costs[i]
    b_i = benefits[i]

    linear_coeff = -b_i + lambda_penalty * (c_i**2 - 2.0 * BUDGET * c_i)

    # In QUBO dictionary, diagonal entries are (i, i)
    Q[(i, i)] = Q.get((i, i), 0.0) + linear_coeff

# Off-diagonal terms: i < j
for i in range(N):
    c_i = costs[i]
    for j in range(i + 1, N):
        c_j = costs[j]
        quad_coeff = 2.0 * lambda_penalty * c_i * c_j
        Q[(i, j)] = Q.get((i, j), 0.0) + quad_coeff

log(f"[Cell 2] QUBO dictionary built with {len(Q)} non-zero entries.")

# Examine at a few entries
list(Q.items())[:10]

[Cell 2] Building QUBO for N = 40 binary variables.
[Cell 2] Using budget BUDGET = 8.00
[Cell 2] Using penalty λ = 10.00
[Cell 2] QUBO dictionary built with 820 non-zero entries.


[((0, 0), np.float64(-280.6)),
 ((1, 1), np.float64(-280.6)),
 ((2, 2), np.float64(-280.6)),
 ((3, 3), np.float64(-280.6)),
 ((4, 4), np.float64(-280.6)),
 ((5, 5), np.float64(-280.6)),
 ((6, 6), np.float64(-280.6)),
 ((7, 7), np.float64(-280.6)),
 ((8, 8), np.float64(-280.6)),
 ((9, 9), np.float64(-280.6))]

### What Cell 2 Just Did

This step constructed the QUBO coefficients corresponding to the trial
selection problem over the `qubo_candidates` set.

Using:

- `benefits[i] = benefit_score` of trial *i*,
- `costs[i] = estimated_trial_cost` of trial *i*,
- A chosen **budget** `BUDGET`, and
- A penalty weight `λ = lambda_penalty`,

the notebook encoded the objective:

\[
\min_x \left(
    - \sum_i b_i x_i
    + \lambda (\sum_i c_i x_i - B)^2
\right)
\]

into a QUBO dictionary `Q` where each key `(i, j)` stores the coefficient on
the term \( x_i x_j \) (with diagonal entries `(i, i)` representing linear
terms \( x_i \)).

This `Q` structure can now be passed to classical QUBO solvers or quantum /
hybrid algorithms (e.g., QAOA) in later notebooks.



In [4]:
# ============================================================
# Cell 3 — Persist QUBO and associated metadata
# ============================================================

import json

QUBO_DIR = Path("data/qubo")
QUBO_DIR.mkdir(parents=True, exist_ok=True)

qubo_path = QUBO_DIR / "scenario_B_qubo.json"
meta_path = QUBO_DIR / "scenario_B_qubo_metadata.csv"

# Save QUBO as a JSON dictionary: keys "i,j" → float coefficient
Q_serializable = {f"{i},{j}": float(v) for (i, j), v in Q.items()}

with qubo_path.open("w") as f:
    json.dump(Q_serializable, f)

log(f"[Cell 3] Wrote QUBO dictionary to {qubo_path}")

# Save candidate trial metadata with variable indices
meta_df = qubo_candidates.copy()
meta_df = meta_df.reset_index().rename(columns={"index": "var_index"})

meta_df.to_csv(meta_path, index=False)
log(f"[Cell 3] Wrote QUBO variable metadata to {meta_path} with shape {meta_df.shape}")

meta_df.head()


[Cell 3] Wrote QUBO dictionary to data/qubo/scenario_B_qubo.json
[Cell 3] Wrote QUBO variable metadata to data/qubo/scenario_B_qubo_metadata.csv with shape (40, 14)


,var_index,nct_id,brief_title,overall_status,phase,conditions,interventions,location_countries,lead_sponsor,lead_sponsor_norm,region_label,estimated_trial_cost,enrollment_feasibility_score,benefit_score
0,0,NCT00003042,Chemotherapy and Stem Cell Transplantation in ...,"Active, not recruiting",Phase 2,['Breast Cancer'],['filgrastim' 'cisplatin' 'cyclophosphamide' '...,['United States'],City of Hope Medical Center,City of Hope Medical Center,Global / Multi-Region,2.0,1.0,0.6
1,1,NCT06234631,Cannabidiol for Postoperative Opioid Reduction...,Recruiting,Phase 2,"['Knee Replacement Surgery' 'Osteoarthritis, K...",['Epidiolex oral solution' 'Placebo'],['United States'],Chad Brummett,Chad Brummett,Global / Multi-Region,2.0,1.0,0.6
2,2,NCT06257537,Sustained Acoustic Medicine for Symptomatic Tr...,Recruiting,Phase 2,['Osteo Arthritis Knee' 'Arthritis'],['Sustained Acoustic Device with 2.5% Diclofen...,['United States'],"ZetrOZ, Inc.","ZetrOZ, Inc.",Global / Multi-Region,2.0,1.0,0.6
3,3,NCT06257875,A Study to Assess Adverse Events and Change in...,"Active, not recruiting",Phase 2,['Ulcerative Colitis'],['Lutikizumab' 'Lutikizumab' 'Adalimumab'],['Australia' 'Austria' 'Belgium' 'Bulgaria' 'C...,AbbVie,AbbVie,Global / Multi-Region,2.0,1.0,0.6
4,4,NCT06259123,Neoadjuvant PSMA-RLT in Oligometastatic PCa,Recruiting,Phase 2,['Prostate Cancer'],['[177Lu]Lu-PSMA I&T'],['Austria'],Medical University of Vienna,Medical University of Vienna,Global / Multi-Region,2.0,1.0,0.6


### What Cell 3 Just Did

This step saved both the QUBO and the mapping from binary variables back to
real clinical trials.

It created:

- `data/qubo/scenario_B_qubo.json`
  - A JSON-serializable dictionary where each key `"i,j"` corresponds to a
    QUBO term \( x_i x_j \) with the given coefficient.
- `data/qubo/scenario_B_qubo_metadata.csv`
  - A table that maps each binary variable `var_index` back to:
    - `nct_id`
    - trial phase, status
    - cost, benefit_score, and any other scenario features.

Together, these files allow future notebooks to:

- Load the exact same QUBO instance, and
- Recover which real-world trials each binary variable represents.

## Notebook Summary — QUBO Design for Trial Selection

In this notebook we translated the Scenario B trial selection problem into a
QUBO formulation suitable for classical or quantum solvers.

Steps:

1. Loaded Scenario B trials and (optionally) the greedy baseline solution.
2. Chose a manageable subset of high-benefit trials (`qubo_candidates`) to
   keep the number of binary variables reasonable.
3. Encoded the objective:
   - Maximize total `benefit_score`
   - Subject to a budget on `estimated_trial_cost`
   as a QUBO:
   \[
   \min_x \left(
       - \sum_i b_i x_i
       + \lambda (\sum_i c_i x_i - B)^2
   \right)
   \]
4. Built a QUBO dictionary `Q[(i, j)]` capturing all linear and quadratic
   coefficients.
5. Persisted:
   - The QUBO dictionary to `data/qubo/scenario_B_qubo.json`
   - The variable→trial mapping to `data/qubo/scenario_B_qubo_metadata.csv`.

This QUBO instance, together with the classical greedy solution from 03a,
provides a clean starting point for future quantum and hybrid experiments in
later Phase 3 notebooks.
